## CapnoBase — validación externa con parámetros congelados desde BIDMC
Todos los parámetros se leen de `results/config_congelada.json`. No se ajusta nada aquí.
Referencia: inicio de espiración marcado en la capnografía (`co2_startexp`).
CapnoBase no tiene segundo anotador: una ventana es válida si tiene referencia y no tiene artefacto
según nuestro detector. Las marcas de artefacto de CapnoBase solo se usan para comparar el detector.

In [ ]:
# c-1 — Imports, configuración congelada y rutas
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import butter, decimate, detrend, filtfilt, resample_poly

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CFG = json.loads((BASE_DIR / "results" / "config_congelada.json").read_text())
print(json.dumps(CFG, indent=1))

CAPNO_DIR = BASE_DIR / "data" / "capnobase" / "csv"     # copia aquí la carpeta csv de CapnoBase
FS = 125                                                 # se remuestrea a la misma fs que BIDMC
VENTANA_S, PASO_S = CFG["ventana_s"], CFG["paso_s"]
RPM_MIN, RPM_MAX = CFG["rpm_min"], CFG["rpm_max"]
casos = sorted(p.name.replace("_signal.csv", "") for p in CAPNO_DIR.glob("*_signal.csv"))
print(len(casos), "casos:", casos[:5], "...")


In [ ]:
# c-2 — Lectura de CapnoBase (formato csv de la revisión r4)
def _lista(celda):
    """En CapnoBase cada celda es una lista de números separados por espacios ("" si está vacía)."""
    if pd.isna(celda) or str(celda).strip() == "":
        return np.array([])
    return np.array(str(celda).split(), dtype=float)

def leer_caso(caso):
    sig = pd.read_csv(CAPNO_DIR / f"{caso}_signal.csv")
    par = pd.read_csv(CAPNO_DIR / f"{caso}_param.csv").iloc[0]
    meta = pd.read_csv(CAPNO_DIR / f"{caso}_meta.csv").iloc[0]
    lab = pd.read_csv(CAPNO_DIR / f"{caso}_labels.csv", keep_default_na=False).iloc[0]
    sf = pd.read_csv(CAPNO_DIR / f"{caso}_SFresults.csv", keep_default_na=False).iloc[0]

    fs0 = int(par["samplingrate_pleth"])                   # 300 Hz
    ppg = resample_poly(sig["pleth_y"].to_numpy(float), FS, fs0)   # 300 -> 125 Hz
    marcas = np.sort(_lista(lab["co2_startexp_x"])) / int(par["samplingrate_co2"])   # muestras -> s
    art = _lista(lab["pleth_artif_x"]) / fs0                # pares [inicio fin] en s (si los hay)
    art = art.reshape(-1, 2) if len(art) % 2 == 0 else np.empty((0, 2))
    sf_t, sf_rr = _lista(sf["x"]), _lista(sf["SmartFusion_y"])
    info = {"Caso": caso, "Edad": meta["subject_age"], "Ventilación": par["case_ventilation"]}
    return ppg, marcas, art, (sf_t, sf_rr), info


In [ ]:
# c-3 — Funciones congeladas (idénticas a BIDMC v2)
def rr_referencia(marcas_s, t0, t1, min_marcas=3):
    m = marcas_s[(marcas_s >= t0) & (marcas_s < t1)]
    return np.nan if len(m) < min_marcas else 60.0 / np.median(np.diff(m))

def rr_fft(segmento, fs, nfft=2048, ancho_rpm=2.0):
    q = int(fs // 5)
    x = decimate(decimate(segmento, 5, zero_phase=True), q // 5, zero_phase=True)
    x = detrend(x) * np.hanning(len(x))
    f = np.fft.rfftfreq(nfft, q / fs) * 60
    P = np.abs(np.fft.rfft(x, nfft)) ** 2
    banda = (f >= RPM_MIN) & (f <= RPM_MAX)
    k = np.argmax(np.where(banda, P, -np.inf))
    return f[k], P[banda & (np.abs(f - f[k]) <= ancho_rpm)].sum() / P[banda].sum()

def evaluar_caso(caso):
    ppg, marcas, art, (sf_t, sf_rr), info = leer_caso(caso)
    b, a = butter(4, CFG["lp_fft_hz"], btype="low", fs=FS)
    riiv_fft = filtfilt(b, a, ppg)
    b, a = butter(4, CFG["lp_art_hz"], btype="low", fs=FS)
    riiv = filtfilt(b, a, ppg)
    n5 = int(5 * FS)
    tipico = np.median([np.std(riiv[i:i + n5]) for i in range(0, len(riiv) - n5 + 1, n5)])
    filas, ventanas = [], []
    for t0 in np.round(np.arange(0, len(ppg) / FS - VENTANA_S + 1e-9, PASO_S), 3):
        t1 = t0 + VENTANA_S
        i0, i1 = int(round(t0 * FS)), int(round(t1 * FS))
        rr, q = rr_fft(riiv_fft[i0:i1], FS)
        exc = max(np.std(riiv[i:i + n5]) for i in range(i0, i1 - n5 + 1, n5)) / tipico
        en_sf = (sf_t >= t0) & (sf_t <= t1) & np.isfinite(sf_rr) if len(sf_rr) else np.array([False])
        w = ppg[i0:i1]
        ventanas.append(((w - w.mean()) / (w.std() + 1e-6)).astype(np.float32))
        filas.append({**info, "Inicio (s)": t0,
                      "RR referencia (rpm)": rr_referencia(marcas, t0, t1),
                      "RR FFT (rpm)": rr, "Calidad FFT": q, "Excursión rel.": exc,
                      "Artefacto CapnoBase": bool(((art[:, 0] < t1) & (art[:, 1] > t0)).any()),
                      "RR SmartFusion (rpm)": np.nanmean(sf_rr[en_sf]) if en_sf.any() else np.nan})
    return pd.DataFrame(filas), np.stack(ventanas)


In [ ]:
# c-4 — FFT en todos los casos
partes = [evaluar_caso(c) for c in casos]
dfc = pd.concat([p[0] for p in partes], ignore_index=True)
Xc = np.concatenate([p[1] for p in partes])
dfc["Artefacto"] = dfc["Excursión rel."] > CFG["umbral_art"]
dfc["Válida"] = ~dfc["Artefacto"] & dfc["RR referencia (rpm)"].notna()
print(f"{len(casos)} casos, {len(dfc)} ventanas, {dfc['Válida'].sum()} válidas "
      f"({100 * dfc['Válida'].mean():.1f} %)")
print("Acuerdo de nuestro detector con las marcas de artefacto de CapnoBase:")
print(pd.crosstab(dfc["Artefacto"], dfc["Artefacto CapnoBase"],
                  rownames=["nuestro"], colnames=["CapnoBase"]))


In [ ]:
# c-5 — CNN-LSTM entrenada solo con BIDMC y regla híbrida congelada
import tensorflow as tf
modelo = tf.keras.models.load_model(BASE_DIR / "results" / "cnn_lstm_bidmc53_final.keras")
dfc["RR CNN (rpm)"] = modelo.predict(Xc[:, :, None], verbose=0).ravel() * 60
borde = (dfc["RR FFT (rpm)"] < RPM_MIN + CFG["borde_rpm"]) | (dfc["RR FFT (rpm)"] > RPM_MAX - CFG["borde_rpm"])
usa_fft = ~borde & (dfc["Calidad FFT"] >= CFG["q_hibrido"])
dfc["RR híbrido (rpm)"] = np.where(usa_fft, dfc["RR FFT (rpm)"], dfc["RR CNN (rpm)"])
print(f"El híbrido usa la FFT en {100 * usa_fft[dfc['Válida']].mean():.1f} % de las ventanas válidas")


In [ ]:
# c-6 — Resultados: MAE por caso, mediana, y por grupo
metodos = {"FFT": "RR FFT (rpm)", "CNN": "RR CNN (rpm)", "Híbrido": "RR híbrido (rpm)",
           "SmartFusion": "RR SmartFusion (rpm)"}
v = dfc[dfc["Válida"]].copy()
for m, col in metodos.items():
    v[f"e_{m}"] = (v[col] - v["RR referencia (rpm)"]).abs()
v["Grupo"] = np.where(v["Edad"] >= 18, "adulto", "niño")

pc = v.groupby(["Caso", "Grupo", "Ventilación"])[[f"e_{m}" for m in metodos]].mean().reset_index()
pc.to_csv(BASE_DIR / "results" / "capnobase_mae_por_caso.csv", index=False)
v.to_csv(BASE_DIR / "results" / "capnobase_ventanas.csv", index=False)

def resumen(g, nombre):
    txt = "  ".join(f"{m}={g[f'e_{m}'].median():.2f}" for m in metodos)
    print(f"{nombre:<28} casos={len(g):2d}  MAE mediano  {txt}")

resumen(pc, "Todos")
for k, g in pc.groupby("Grupo"):
    resumen(g, f"Grupo: {k}")
for k, g in pc.groupby("Ventilación"):
    resumen(g, f"Ventilación: {k}")
print(f"\nVentanas con referencia > {RPM_MAX} rpm (fuera de banda): "
      f"{(v['RR referencia (rpm)'] > RPM_MAX).sum()} de {len(v)}")
print("SmartFusion: ventanas sin estimación (el método se abstiene):",
      v["RR SmartFusion (rpm)"].isna().sum())

from scipy.stats import wilcoxon
for a_, b_ in [("Híbrido", "FFT"), ("Híbrido", "CNN"), ("FFT", "CNN")]:
    print(f"Wilcoxon {a_} vs {b_}: p = {wilcoxon(pc[f'e_{a_}'], pc[f'e_{b_}']).pvalue:.4f}")
